# Pose Estimator Test
This file is a test script of MediaPipe's Pose Estimator on pre-recorded video.

In [ ]:
import numpy as np
import pandas as pd
import cv2
import os
from enum import IntEnum

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles

In [ ]:
def draw_landmarks_on_image(rgb_image, detection_result):
    pose_landmarks = detection_result.pose_landmarks[0]
    annotated_image = np.copy(rgb_image)

    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)
    h, w, _ = annotated_image.shape

    if pose_landmarks:
        drawing_utils.draw_landmarks(
            image=annotated_image,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=pose_landmark_style,
            connection_drawing_spec=pose_connection_style,
        )

        for idx, lm in enumerate(pose_landmarks):
            cx, cy = int(lm.x * w), int(lm.y * h)

            # Write the ID number next to the landmark
            cv2.putText(
                annotated_image,
                str(idx),
                (cx + 5, cy + 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                1,
            )

    return annotated_image


if not os.path.exists("pose_landmarker_heavy.task"):
    import urllib.request

    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task"
    urllib.request.urlretrieve(url, "pose_landmarker_heavy.task")

# STEP 2: Create an PoseLandmarker object.
base_options = python.BaseOptions(model_asset_path="pose_landmarker_heavy.task")
options = vision.PoseLandmarkerOptions(
    base_options=base_options, output_segmentation_masks=True
)
detector = vision.PoseLandmarker.create_from_options(options)

In [ ]:
class LANDMARKS(IntEnum):
    """MediaPipe pose landmark indices (0–32)."""
    NOSE = 0
    LEFT_EYE_INNER = 1
    LEFT_EYE = 2
    LEFT_EYE_OUTER = 3
    RIGHT_EYE_INNER = 4
    RIGHT_EYE = 5
    RIGHT_EYE_OUTER = 6
    LEFT_EAR = 7
    RIGHT_EAR = 8
    MOUTH_LEFT = 9
    MOUTH_RIGHT = 10
    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12
    LEFT_ELBOW = 13
    RIGHT_ELBOW = 14
    LEFT_WRIST = 15
    RIGHT_WRIST = 16
    LEFT_PINKY = 17
    RIGHT_PINKY = 18
    LEFT_INDEX = 19
    RIGHT_INDEX = 20
    LEFT_THUMB = 21
    RIGHT_THUMB = 22
    LEFT_HIP = 23
    RIGHT_HIP = 24
    LEFT_KNEE = 25
    RIGHT_KNEE = 26
    LEFT_ANKLE = 27
    RIGHT_ANKLE = 28
    LEFT_HEEL = 29
    RIGHT_HEEL = 30
    LEFT_FOOT_INDEX = 31
    RIGHT_FOOT_INDEX = 32

In [ ]:
def calculate_angle(a, b, c):
    ba = np.array([a.x - b.x, a.y - b.y, a.z - b.z])
    bc = np.array([c.x - b.x, c.y - b.y, c.z - b.z])

    mags = np.linalg.norm(ba) * np.linalg.norm(bc)
	
    if mags < 1e-6: # avoid near-zero instability 
        return 0.0

    cos_angle = np.dot(ba, bc) / mags
    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    return np.degrees(np.arccos(cos_angle))


def estimate_joint_angles(detection_result):
    if not detection_result.pose_world_landmarks:
        return {}

    landmarks = detection_result.pose_world_landmarks[0]  # assume one person

    L = LANDMARKS
    angles = {}

    angles['left_elbow'] = calculate_angle(
		landmarks[L.LEFT_SHOULDER],
		landmarks[L.LEFT_ELBOW], 
		landmarks[L.LEFT_WRIST]
	)
    angles['right_elbow'] = calculate_angle(
		landmarks[L.RIGHT_SHOULDER], 
		landmarks[L.RIGHT_ELBOW], 
		landmarks[L.RIGHT_WRIST]
	)
    angles['left_knee'] = calculate_angle(
		landmarks[L.LEFT_HIP], 
		landmarks[L.LEFT_KNEE], 
		landmarks[L.LEFT_ANKLE]
	)
    angles['right_knee'] = calculate_angle(
		landmarks[L.RIGHT_HIP], 
		landmarks[L.RIGHT_KNEE], 
		landmarks[L.RIGHT_ANKLE]
	)

    return angles


def display_angles_on_image(image, angles):
    """Display angle values as text on the image"""
    y_offset = 30
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    color = (0, 0, 255)
    thickness = 2

    for joint_name, angle_value in angles.items():
        text = f"{joint_name}: {angle_value:.2f}°"
        cv2.putText(image, text, (10, y_offset), font, font_scale, color, thickness)
        y_offset += 30

    return image

In [ ]:
# Live feed

cap = cv2.VideoCapture(1)

if not cap.isOpened():
    raise RuntimeError("Webcam could not be opened.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Frame grab failed.")
        break

    # BGR to RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    detection_result = detector.detect(image)

    if detection_result.pose_world_landmarks:
        # Estimate joint angles
        angles = estimate_joint_angles(detection_result)

        landmark_image = draw_landmarks_on_image(image.numpy_view(), detection_result)

        # Display angles on the image
        landmark_image = display_angles_on_image(landmark_image, angles)

        output = cv2.cvtColor(landmark_image, cv2.COLOR_RGB2BGR)

        cv2.imshow("live pose estimation", output)

    # q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()